In [11]:
import numpy as np
import pandas as pd
import pywt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)


In [17]:
# ---------- 0. Settings ----------
raw_interval_min = 1
bin_min = 5
dt = bin_min / 60                          # hours per row after binning
per_day = 24 * 60 // bin_min               # 288
wavelet = "cmor1.5-1.0"
periods = np.geomspace(1, 39, 100)
scales = periods / (pywt.central_frequency(wavelet) * dt)
cr_rows = (periods >= 23) & (periods <= 25)     # circadian band
ur_rows = (periods >= 1) & (periods <= 3)       # ultradian band

# ---------- 1. Load ----------
female_temp = pd.read_csv('../data/inputs/fem_temp.csv')
female_act  = pd.read_csv('../data/inputs/fem_act.csv')

# ---------- 2. Clean and bin ----------
def prep_temp(df):
    df = df.select_dtypes("number").reset_index(drop=True)
    df = df.interpolate(limit=10).bfill().ffill()
    df = df.clip(lower=35)                                    # floor at 35 C
    mu, sd = df.mean(), df.std()
    df = df.clip(lower=mu - 3*sd, upper=mu + 3*sd, axis=1)    # cap at 3 SD
    return df.groupby(df.index // (bin_min // raw_interval_min)).mean()

def bin_act(df):
    df = df.select_dtypes("number").reset_index(drop=True).fillna(0)
    return df.groupby(df.index // (bin_min // raw_interval_min)).mean()

f = prep_temp(female_temp)
act5 = bin_act(female_act)
n = min(len(f), len(act5))
f, act5 = f.iloc[:n], act5.iloc[:n]
assert f.shape[1] == act5.shape[1], "temp and activity sheets have different numbers of mice"
mice = f.shape[1]
print("rows:", n, "| mice:", mice, "| days:", round(n / per_day, 1))   # expect 4032, 13, 14.0

# ---------- 3. Activity-corrected CBT (linear regression residual) ----------
resid = {}
for col in f.columns:
    a = np.log1p(act5[col].values)
    X = np.column_stack([np.roll(a, k) for k in (0, 1, 2, 3)])   # activity now + 3 lags (15 min)
    y_t = f[col].values
    resid[col] = y_t - LinearRegression().fit(X, y_t).predict(X) + y_t.mean()
resid = pd.DataFrame(resid, index=f.index)

# ---------- 4. Wavelet band amplitude on activity-corrected CBT ----------
def band_series(df):
    circ, ultr = [], []
    for col in df.columns:
        x = df[col].values - df[col].values.mean()
        coefs, _ = pywt.cwt(x, scales, wavelet, sampling_period=dt)
        amp = np.abs(coefs)[:, per_day : 13 * per_day]      # keep days 2-13
        circ.append(amp[cr_rows].max(axis=0))
        ultr.append(amp[ur_rows].max(axis=0))
    return np.array(circ), np.array(ultr)

circ_adj, ultr_adj = band_series(resid)

# ---------- 5. Feature table: one row per mouse per day ----------
def days(x):                                   # (mice, 12*per_day) -> (mice, 12, per_day)
    return x.reshape(mice, 12, per_day)

U_adj, C_adj = days(ultr_adj), days(circ_adj)
T_raw = days(f.values[per_day:13*per_day].T)
T_adj = days(resid.values[per_day:13*per_day].T)
A_log = days(np.log1p(act5.values)[per_day:13*per_day].T)
w = slice(72, 144)                             # hours 6-12 in 5-min bins

rows = []
for i, col in enumerate(f.columns):
    for d in range(12):
        adj = T_adj[i, d]
        below = np.where(adj < np.median(adj))[0]
        rows.append({
            "mouse_id": col,
            "day_index": d + 2,                # 2..13
            "vivian__adj_ultradian_amp": U_adj[i, d].mean(),
            "vivian__adj_circadian_amp": C_adj[i, d].mean(),
            "vivian__adj_mean_h6_12":    adj[w].mean(),
            "vivian__adj_range":         adj.max() - adj.min(),
            "vivian__adj_decline_hour":  below[0] * dt if len(below) else np.nan,
            "vivian__act_cbt_corr":      np.corrcoef(A_log[i, d], T_raw[i, d])[0, 1],
        })

vivian = pd.DataFrame(rows)
vivian["estrus"] = ((vivian.day_index - 2) % 4 == 0).astype(int)   # days 2, 6, 10 = estrus (assumed)
feature_cols = [c for c in vivian.columns if c.startswith("vivian__")]
vivian[feature_cols] = vivian[feature_cols].fillna(vivian[feature_cols].median())
vivian.to_csv("outputs/vivian_features.csv", index=False)         # uncentered, for the shared pipeline

print(vivian.shape)                            # expect (156, 9)
print(vivian.head(12).round(3))
print("\nCorrelation among features (pairs above ~0.9 are redundant):")
print(vivian[feature_cols].corr().round(2))

# ---------- 6. Classifier setup (logistic regression, leave-one-mouse-out) ----------
CENTER_PER_MOUSE = True      # for this analysis only; the CSV above is uncentered
X_df = vivian[feature_cols].copy()
if CENTER_PER_MOUSE:
    X_df = X_df - X_df.groupby(vivian.mouse_id).transform("mean")

y = vivian["estrus"].values
groups = vivian["mouse_id"].values

model = make_pipeline(StandardScaler(),
                      LogisticRegression(class_weight="balanced", max_iter=1000))
logo = LeaveOneGroupOut()

rng = np.random.default_rng(0)
def shuffled_labels():
    return np.concatenate([rng.permutation(y[groups == g]) for g in pd.unique(groups)])

sets = {
    "adjusted wavelet":   ["vivian__adj_ultradian_amp", "vivian__adj_circadian_amp"],
    "adjusted summaries": ["vivian__adj_mean_h6_12", "vivian__adj_range", "vivian__adj_decline_hour"],
    "coupling":           ["vivian__act_cbt_corr"],
    "all six":            feature_cols,
}

# ---------- 7. Metrics ----------
rows_out = []
for name, cols in sets.items():
    Xm = X_df[cols].values
    p = cross_val_predict(model, Xm, y, groups=groups, cv=logo, method="predict_proba")[:, 1]
    pred = (p > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    null = []
    for _ in range(20):
        ys = shuffled_labels()
        ps = cross_val_predict(model, Xm, ys, groups=groups, cv=logo, method="predict_proba")[:, 1]
        null.append(roc_auc_score(ys, ps))
    rows_out.append({
        "feature set": name,
        "AUC": roc_auc_score(y, p),
        "AUC shuffled (mean)": np.mean(null),
        "AUC shuffled (max)": np.max(null),
        "accuracy": accuracy_score(y, pred),
        "balanced acc": balanced_accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred),
        "specificity": tn / (tn + fp),
        "F1": f1_score(y, pred),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    })

res = pd.DataFrame(rows_out)
print("\n", res.round(2).to_string(index=False))
print("\nchance AUC = 0.50; always guessing 'not estrus' = %.0f%% accuracy" % (100 * (1 - y.mean())))
res.round(3).to_csv("outputs/estrus_classifier_results.csv", index=False)

# ---------- 8. Coefficients (all six, standardized) ----------
full = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
full.fit(X_df[feature_cols].values, y)
coef = pd.Series(full[-1].coef_[0], index=[c.replace("vivian__", "") for c in feature_cols])
print("\nStandardized coefficients (positive = higher on estrus days):")
print(coef.sort_values(key=np.abs, ascending=False).round(2).to_string())

rows: 4032 | mice: 13 | days: 14.0
(156, 9)
   mouse_id  day_index  vivian__adj_ultradian_amp  vivian__adj_circadian_amp  \
0        f1          2                      0.491                      2.923   
1        f1          3                      0.527                      2.908   
2        f1          4                      0.415                      2.737   
3        f1          5                      0.602                      2.285   
4        f1          6                      0.541                      2.072   
5        f1          7                      0.597                      2.377   
6        f1          8                      0.622                      2.677   
7        f1          9                      0.445                      2.724   
8        f1         10                      0.440                      2.502   
9        f1         11                      0.448                      2.239   
10       f1         12                      0.565                      2.121

In [18]:
for name, cols in sets.items():
    p = cross_val_predict(model, X_df[cols].values, y, groups=groups,
                          cv=logo, method="predict_proba")[:, 1]
    pred = (p > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    print(f"{name:18s} AUC {roc_auc_score(y, p):.2f} | acc {accuracy_score(y, pred):.2f} "
          f"| bal acc {balanced_accuracy_score(y, pred):.2f} | precision {precision_score(y, pred, zero_division=0):.2f} "
          f"| recall {recall_score(y, pred):.2f} | specificity {tn/(tn+fp):.2f} | F1 {f1_score(y, pred):.2f} "
          f"| TP {tp} FP {fp} FN {fn} TN {tn}")

adjusted wavelet   AUC 0.64 | acc 0.60 | bal acc 0.62 | precision 0.34 | recall 0.67 | specificity 0.57 | F1 0.45 | TP 26 FP 50 FN 13 TN 67
adjusted summaries AUC 0.94 | acc 0.86 | bal acc 0.86 | precision 0.67 | recall 0.87 | specificity 0.85 | F1 0.76 | TP 34 FP 17 FN 5 TN 100
coupling           AUC 0.52 | acc 0.56 | bal acc 0.54 | precision 0.28 | recall 0.49 | specificity 0.59 | F1 0.36 | TP 19 FP 48 FN 20 TN 69
all six            AUC 0.97 | acc 0.93 | bal acc 0.93 | precision 0.82 | recall 0.92 | specificity 0.93 | F1 0.87 | TP 36 FP 8 FN 3 TN 109


rows: 4032 | mice: 13 | days: 14.0
(156, 9)
   mouse_id  day_index  vivian__adj_ultradian_amp  vivian__adj_circadian_amp  \
0        f1          2                      0.491                      2.923   
1        f1          3                      0.527                      2.908   
2        f1          4                      0.415                      2.737   
3        f1          5                      0.602                      2.285   
4        f1          6                      0.541                      2.072   
5        f1          7                      0.597                      2.377   
6        f1          8                      0.622                      2.677   
7        f1          9                      0.445                      2.724   
8        f1         10                      0.440                      2.502   
9        f1         11                      0.448                      2.239   
10       f1         12                      0.565                      2.121